In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
# Chỉ cài thư viện cần cho CLIPScore.
%pip install -q "sentence-transformers>=5.1,<6" pillow tqdm

In [ ]:
import csv
import json
import time
from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm

def status(message):
    print(f"[{time.strftime('%H:%M:%S')}] {message}", flush=True)

# Upload thư mục ảnh (gồm cả PNG và JSON) lên MyDrive; có thể chứa 50 ảnh hoặc nhiều hơn.
# Nếu đặt ở nơi khác, sửa đường dẫn này cho đúng.
IMAGE_DIR = Path('/content/drive/MyDrive/20-images')
RUN_DIR = Path('/content/drive/MyDrive/heim_clipscore_results')
if not IMAGE_DIR.is_dir():
    raise FileNotFoundError(f"Không tìm thấy thư mục ảnh: {IMAGE_DIR}. Hãy upload thư mục và sửa IMAGE_DIR.")

images = sorted(p for p in IMAGE_DIR.iterdir() if p.is_file() and p.suffix.lower() == '.png')
if not images:
    raise ValueError(f"Không tìm thấy ảnh PNG trong {IMAGE_DIR}")
jobs = []
errors = []
for image_path in images:
    metadata_path = image_path.with_suffix('.json')
    try:
        metadata = json.loads(metadata_path.read_text(encoding='utf-8-sig'))
        prompt = metadata.get('prompt_vi')
        if not isinstance(prompt, str) or not prompt.strip():
            raise ValueError('thiếu prompt_vi')
        if metadata.get('status') != 'success':
            raise ValueError('status không phải success')
        if metadata.get('case_id') != image_path.stem:
            raise ValueError('case_id không khớp tên ảnh')
        with Image.open(image_path) as image:
            image.load()  # Đọc đầy đủ dữ liệu để phát hiện ảnh bị cắt/hỏng.
        jobs.append({'case_id': metadata['case_id'], 'image': image_path.name, 'prompt': prompt.strip()})
    except (OSError, ValueError, TypeError, AttributeError) as error:
        errors.append(f'{image_path.name}: {error}')
RUN_DIR.mkdir(parents=True, exist_ok=True)
error_path = RUN_DIR / 'anh_loi.csv'
with error_path.open('w', encoding='utf-8-sig', newline='') as file:
    writer = csv.writer(file)
    writer.writerow(['chi_tiet_loi'])
    writer.writerows([error] for error in errors)
if errors:
    print(f'Bỏ qua {len(errors)} ảnh lỗi hoặc thiếu metadata:')
    for error in errors:
        print(' -', error)
    print(f'Danh sách lỗi: {error_path}')
if not jobs:
    raise ValueError('Không có ảnh hợp lệ để chấm. Hãy kiểm tra danh sách lỗi ở trên.')
status(f'Tìm thấy {len(images)} ảnh PNG; sẽ chấm {len(jobs)} ảnh hợp lệ trong {IMAGE_DIR}')

## Chấm tự động

Upload thư mục ảnh lên Google Drive, giữ các file PNG và JSON cùng nhau, rồi chỉnh `IMAGE_DIR` ở ô trên nếu cần. Mỗi ảnh được chấm với `prompt_vi` trong file JSON cùng tên. CLIPScore = 100 × max(cosine, 0); điểm là độ tương đồng, không phải tỷ lệ đúng. Kết quả lưu trong `MyDrive/heim_clipscore_results`.
Notebook tự nhận số ảnh PNG, bỏ qua ảnh hoặc metadata lỗi và ghi danh sách lỗi vào `anh_loi.csv`. Số ảnh thực sự được chấm được in ra khi chạy.


In [ ]:
status('Đang nạp thư viện chấm CLIPScore')
from sentence_transformers import SentenceTransformer

status('[1/2] Đang tải/nạp model CLIP mã hóa ảnh')
image_model = SentenceTransformer('sentence-transformers/clip-ViT-B-32')
status('[2/2] Đang tải/nạp model CLIP đa ngôn ngữ mã hóa prompt')
text_model = SentenceTransformer('sentence-transformers/clip-ViT-B-32-multilingual-v1')
results = []
with tqdm(total=len(jobs), desc='Ảnh đã chấm CLIP', unit=' ảnh', dynamic_ncols=True) as progress:
    for start in range(0, len(jobs), 16):
        batch = jobs[start:start + 16]
        images = []
        for job in batch:
            with Image.open(IMAGE_DIR / job['image']) as img:
                images.append(img.convert('RGB'))
        image_vectors = image_model.encode(images, normalize_embeddings=True, show_progress_bar=False)
        text_vectors = text_model.encode([job['prompt'] for job in batch], normalize_embeddings=True, show_progress_bar=False)
        scores = 100 * (image_vectors * text_vectors).sum(axis=1).clip(min=0, max=1)
        results.extend({**job, 'clipscore': float(score)} for job, score in zip(batch, scores))
        progress.update(len(batch))

output_path = RUN_DIR / f'clipscore_{len(results)}_images.csv'
with output_path.open('w', encoding='utf-8-sig', newline='') as file:
    writer = csv.DictWriter(file, fieldnames=['case_id', 'image', 'prompt', 'clipscore'])
    writer.writeheader()
    writer.writerows(results)
from statistics import mean
print(f"CLIPScore trung bình ({len(results)} ảnh): {mean(row['clipscore'] for row in results):.2f}")
status(f'Đã lưu kết quả: {output_path}')

In [ ]:
# Ô 1/2: đọc và ghép 50 cặp ảnh được tạo từ prompt Việt / Anh.
# Có thể chạy riêng hai ô mới này, không cần chạy các ô Colab phía trên.
%pip install -q "sentence-transformers>=5.1,<6" pillow tqdm

import csv
import json
from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm

PAIR_IMAGE_DIR = Path('/home/hoang/experiments/HELM-demo/outputs/50-images')
# Nếu chạy trên Colab, sửa PAIR_IMAGE_DIR thành thư mục đã upload lên Drive.
PAIR_RESULT_DIR = PAIR_IMAGE_DIR / 'clip_image_similarity_results'
EXPECTED_PAIR_COUNT = 50
if not PAIR_IMAGE_DIR.is_dir():
    raise FileNotFoundError(f'Không tìm thấy thư mục: {PAIR_IMAGE_DIR}')

pair_jobs = []
pair_issues = []
for number in range(1, EXPECTED_PAIR_COUNT + 1):
    case_id = f'COCO14-{number:03d}'
    vi_path = PAIR_IMAGE_DIR / f'{case_id}.png'
    en_path = PAIR_IMAGE_DIR / f'{case_id}-eng.png'
    missing = [p.name for p in (vi_path, en_path) if not p.is_file()]
    if missing:
        pair_issues.append({'case_id': case_id, 'issue': 'Thiếu ảnh: ' + ', '.join(missing)})
        continue
    try:
        for path in (vi_path, en_path):
            with Image.open(path) as image:
                image.load()
        pair_jobs.append({'case_id': case_id, 'image_vi': vi_path.name, 'image_en': en_path.name})
    except (OSError, ValueError) as error:
        pair_issues.append({'case_id': case_id, 'issue': str(error)})

PAIR_RESULT_DIR.mkdir(parents=True, exist_ok=True)
with (PAIR_RESULT_DIR / 'missing_or_invalid_pairs.csv').open('w', encoding='utf-8-sig', newline='') as file:
    writer = csv.DictWriter(file, fieldnames=['case_id', 'issue'])
    writer.writeheader()
    writer.writerows(pair_issues)
print(f'Đủ ảnh và đọc được: {len(pair_jobs)}/{EXPECTED_PAIR_COUNT} cặp.')
for issue in pair_issues:
    print(f"  {issue['case_id']}: {issue['issue']}")
if pair_issues:
    print('Ảnh đang được tạo: chạy lại cả hai ô này khi đã có đủ 50 cặp.')
if not pair_jobs:
    raise ValueError('Chưa có cặp ảnh hợp lệ để so sánh.')


In [ ]:
# Ô 2/2: so sánh ảnh Việt với ảnh Anh bằng embedding ảnh của CLIP.
# cosine = dot hai vector chuẩn hóa; điểm = 100 * max(cosine, 0).
# Đây là CLIP image-image similarity, không phải CLIPScore ảnh-văn bản
# và không phải phần trăm chính xác / xác suất hai ảnh giống nhau.
import numpy as np
from sentence_transformers import SentenceTransformer

pair_clip_model = SentenceTransformer('sentence-transformers/clip-ViT-B-32')
pair_results = []
PAIR_BATCH_SIZE = 8
with tqdm(total=len(pair_jobs), desc='So sánh CLIP Việt–Anh', unit=' cặp') as progress:
    for start in range(0, len(pair_jobs), PAIR_BATCH_SIZE):
        batch = pair_jobs[start:start + PAIR_BATCH_SIZE]
        batch_images = []
        try:
            for job in batch:
                for key in ('image_vi', 'image_en'):
                    with Image.open(PAIR_IMAGE_DIR / job[key]) as image:
                        batch_images.append(image.convert('RGB'))
            vectors = pair_clip_model.encode(
                batch_images, batch_size=PAIR_BATCH_SIZE * 2,
                normalize_embeddings=True, convert_to_numpy=True,
                show_progress_bar=False,
            )
            cosines = np.clip((vectors[0::2] * vectors[1::2]).sum(axis=1), -1.0, 1.0)
            for job, cosine in zip(batch, cosines):
                pair_results.append({**job, 'clip_cosine': float(cosine),
                                     'clip_image_similarity_100': 100.0 * max(float(cosine), 0.0)})
        finally:
            for image in batch_images:
                image.close()
        progress.update(len(batch))

pair_csv_path = PAIR_RESULT_DIR / 'clip_image_similarity_vi_en.csv'
with pair_csv_path.open('w', encoding='utf-8-sig', newline='') as file:
    writer = csv.DictWriter(file, fieldnames=[
        'case_id', 'image_vi', 'image_en', 'clip_cosine', 'clip_image_similarity_100'])
    writer.writeheader()
    writer.writerows(pair_results)

pair_scores = np.array([row['clip_image_similarity_100'] for row in pair_results])
pair_summary = {
    'model': 'sentence-transformers/clip-ViT-B-32',
    'metric': 'CLIP image-image similarity: 100 * max(cosine, 0)',
    'expected_pairs': EXPECTED_PAIR_COUNT,
    'scored_pairs': len(pair_results),
    'complete': len(pair_results) == EXPECTED_PAIR_COUNT,
    'mean': float(pair_scores.mean()),
    'median': float(np.median(pair_scores)),
    'min': float(pair_scores.min()),
    'max': float(pair_scores.max()),
}
pair_summary_path = PAIR_RESULT_DIR / 'clip_image_similarity_summary.json'
pair_summary_path.write_text(json.dumps(pair_summary, ensure_ascii=False, indent=2), encoding='utf-8')
print(f"Đã chấm: {len(pair_results)}/{EXPECTED_PAIR_COUNT} cặp")
print(f"Điểm tương đồng trung bình (0–100): {pair_summary['mean']:.2f}")
print(f"Trung vị: {pair_summary['median']:.2f}; thấp nhất: {pair_summary['min']:.2f}; cao nhất: {pair_summary['max']:.2f}")
if not pair_summary['complete']:
    print('Kết quả tạm thời; chạy lại cả hai ô khi tạo xong ảnh để tổng hợp đủ 50 cặp.')
print(f'Chi tiết từng cặp: {pair_csv_path}')
print(f'Tổng hợp: {pair_summary_path}')
